# Example 2.4 / 3: Polars at the Data Boundary

Companion notebook for **2.4 Data from pandas, NumPy and Files** in the AMPL lab Documentation, section "Polars at the Data Boundary". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Polars at the Data Boundary

Use Polars for scans, joins, filters, and aggregations. Convert only the final shaped table:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                # products
param rate {PROD} > 0;   # tons per hour
param profit {PROD};     # profit per ton

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
import polars as pl

products_pl = pl.DataFrame({
    "PROD": ["bands", "coils"],
    "rate": [200, 140],
    "profit": [25, 30],
})

products_pd = products_pl.to_pandas().set_index("PROD")
ampl.set_data(products_pd, "PROD")

In [5]:
# run.py
print(ampl.set["PROD"].to_list())
print(ampl.param["rate"].to_dict(), ampl.param["profit"].to_dict())

['bands', 'coils']
{'bands': 200, 'coils': 140} {'bands': 25, 'coils': 30}


### Pure AMPL: 

`to_pandas()` needs the `pyarrow` package as well as `polars`
(`pip install polars pyarrow`); without it the conversion fails with
`ModuleNotFoundError: No module named 'pyarrow'`. The
[AMPL book, section 9.6](https://amplbook.readthedocs.io/en/latest/notebooks/09/9_6_data_in_polars.html)
loads the diet model's food and nutrient tables the same way.

The index-setting step is essential: without it, the pandas frame has a default integer index,
and those integers would become the set members.

Perform key-preserving operations before conversion and select only model-owned columns at the
boundary:

In [6]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [7]:
# data.py
import polars as pl

products_pl = pl.DataFrame({
    "PROD": ["bands", "coils"],
    "rate": [200, 140],
    "profit": [25, 30],
})

products_pd = products_pl.to_pandas().set_index("PROD")
ampl.set_data(products_pd, "PROD")

In [8]:
# run.py
print(ampl.set["PROD"].to_list())
print(ampl.param["rate"].to_dict(), ampl.param["profit"].to_dict())

['bands', 'coils']
{'bands': 200, 'coils': 140} {'bands': 25, 'coils': 30}
